# Session 2: three circles pin down the epicentre
**A. Real-world problem.** One station tells you only HOW FAR the quake was, not in which
direction. Where exactly was the epicentre (the point on the ground above the source)?

**Plan.** (1) where do vp and vs come from, (2) distance from the S-P time, (3) travel-time
curves, (4) three circles meet at the epicentre (trilateration), (5) why two stations are not enough.
We work on a flat map in kilometres. Opengps uses the same idea with satellites; openemwave
and openoscillator cover waves and vibrations in general.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

VP, VS = 6.0, 3.5   # APPROXIMATE typical crust speeds, km/s

# a made-up network (km) and a made-up quake, so we know the right answer
STATIONS = np.array([[0.0, 0.0], [100.0, 0.0], [40.0, 90.0]])
QUAKE = np.array([60.0, 35.0])

## Step 1: why vs is smaller than vp
In a uniform elastic solid the speeds depend on how the material resists squeezing and
shearing. The ratio depends only on Poisson's ratio nu:  vp / vs = sqrt(2 (1 - nu) / (1 - 2 nu)).
For nu = 0.25 this is sqrt(3) = 1.732. Our 6.0 / 3.5 = 1.714 is close: real rock is not
an ideal solid, so we keep the speeds as approximate.

In [ ]:
def vp_vs_ratio(nu):
    """vp / vs for a uniform elastic solid with Poisson's ratio nu (0 <= nu < 0.5)."""
    nu = float(nu)
    if not 0.0 <= nu < 0.5:
        raise ValueError("Poisson's ratio must be in [0, 0.5)")
    return float(np.sqrt(2.0 * (1.0 - nu) / (1.0 - 2.0 * nu)))

## Step 2: distance from the S-P time, and travel-time curves
P arrives at tp = t0 + d / vp and S at ts = t0 + d / vs. Subtract: the unknown origin time
t0 cancels, so  d = (ts - tp) * vp * vs / (vp - vs).

In [ ]:
def distance_sp(ts, tp, vp=VP, vs=VS):
    """Distance (km) from the S and P arrival times (s) at one station."""
    return (np.asarray(ts, dtype=float) - np.asarray(tp, dtype=float)) * vp * vs / (vp - vs)


def travel_time_curves(d, vp=VP, vs=VS):
    """P and S travel times (s) at distances d (km): two straight lines through the origin."""
    d = np.asarray(d, dtype=float)
    return d / vp, d / vs

## Step 3: three circles
Each station gives a circle of radius r_i around it. Writing the circle equation
(x - xi)^2 + (y - yi)^2 = ri^2 for each station and subtracting the first from the others
cancels x^2 + y^2 and leaves STRAIGHT lines in x and y:
    2 (xi - x1) x + 2 (yi - y1) y = r1^2 - ri^2 + xi^2 + yi^2 - x1^2 - y1^2.
Two such lines meet in one point.

In [ ]:
def trilaterate(stations, radii):
    """Epicentre (x, y) from >= 3 stations and their distances (km), by solving the linear system."""
    s = np.asarray(stations, dtype=float)
    r = np.asarray(radii, dtype=float)
    A = 2.0 * (s[1:] - s[0])
    b = r[0] ** 2 - r[1:] ** 2 + np.sum(s[1:] ** 2, axis=1) - np.sum(s[0] ** 2)
    if np.linalg.matrix_rank(A) < 2:
        raise ValueError("stations are on one line: the epicentre is not determined")
    return np.linalg.lstsq(A, b, rcond=None)[0]


def circle_intersections(c1, r1, c2, r2):
    """The 0, 1 or 2 points where two circles meet (list of (x, y))."""
    c1, c2 = np.asarray(c1, dtype=float), np.asarray(c2, dtype=float)
    d = float(np.linalg.norm(c2 - c1))
    if d == 0.0 or d > r1 + r2 or d < abs(r1 - r2):
        return []
    a = (r1 ** 2 - r2 ** 2 + d ** 2) / (2.0 * d)
    h2 = r1 ** 2 - a ** 2
    mid = c1 + a * (c2 - c1) / d
    if h2 <= 1e-12:
        return [(float(mid[0]), float(mid[1]))]
    h = np.sqrt(h2)
    perp = np.array([-(c2 - c1)[1], (c2 - c1)[0]]) / d
    p1, p2 = mid + h * perp, mid - h * perp
    return [(float(p1[0]), float(p1[1])), (float(p2[0]), float(p2[1]))]


def station_distances(stations, quake):
    """True distance (km) from the quake to each station."""
    return np.linalg.norm(np.asarray(stations, dtype=float) - np.asarray(quake, dtype=float), axis=1)

## Step 4: see it
Left: the three circles all pass through the epicentre. Right: the travel-time lines;
the vertical distance between them is the S-P gap.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"vp/vs for nu = 0.25: {vp_vs_ratio(0.25):.3f}; our 6.0/3.5 = {VP / VS:.3f}")
    d_true = station_distances(STATIONS, QUAKE)
    tp, ts = travel_time_curves(d_true)
    print("true distances (km):", d_true.round(2))
    print("S-P gaps (s):", (ts - tp).round(2))
    est = trilaterate(STATIONS, distance_sp(ts, tp))
    print("recovered epicentre (km):", est.round(3), " true:", QUAKE)
    pts = circle_intersections(STATIONS[0], d_true[0], STATIONS[1], d_true[1])
    print("two stations give two points:", [tuple(round(c, 2) for c in p) for p in pts])

    fig, ax = plt.subplots(figsize=(5.6, 5.4))
    th = np.linspace(0, 2 * np.pi, 200)
    for (sx, sy), r, name in zip(STATIONS, d_true, "ABC"):
        ax.plot(sx + r * np.cos(th), sy + r * np.sin(th), lw=1, alpha=0.7)
        ax.plot([sx], [sy], "^", color="k")
        ax.annotate(f"station {name}", (sx, sy), textcoords="offset points", xytext=(5, 6))
    ax.plot(*QUAKE, "r*", ms=16, label="epicentre (true)")
    ax.plot(*pts[1], "x", color="gray", ms=10, label="mirror point from A and B only")
    ax.set_xlim(-30, 150)
    ax.set_ylim(-60, 150)
    ax.set_aspect("equal")
    ax.set_xlabel("east (km)")
    ax.set_ylabel("north (km)")
    ax.set_title("Three circles meet at the epicentre")
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "three_circles.png", dpi=120)

    dd = np.linspace(0, 200, 41)
    p_t, s_t = travel_time_curves(dd)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(dd, p_t, label="P wave (6.0 km/s)")
    ax2.plot(dd, s_t, label="S wave (3.5 km/s)")
    ax2.fill_between(dd, p_t, s_t, alpha=0.15, label="S-P gap")
    ax2.set_xlabel("distance to the quake (km)")
    ax2.set_ylabel("travel time (s)")
    ax2.set_title("Travel-time curves: the gap grows with distance")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "travel_times.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Move the quake in `lesson.py` to (80, 60) km, keep the stations and print the recovered epicentre. Predict it first.
2. **Stretch.** Write `unique(stations, quake)` that returns True when three stations give a unique epicentre, using the rank of the matrix of differences.
3. **Explain.** Explain why the S-P method does not need the origin time of the quake, and what it would need if we used only P arrivals (hint: Session 3).

## Assessment
- **A. Conceptual.** Why do two stations leave two possible epicentres, and why does a third station fix this? Why do three stations on one straight line still fail?
- **B. Numerical.** Compute the ratio vp/vs for Poisson's ratio 0.25 and compare it with 6.0 / 3.5. Which Poisson's ratio gives exactly 6.0 / 3.5?
- **C. Graph and simulation interpretation.** In the app, set the quake to (60, 35) km and add a clock error of +1.0 s at station A. Describe how the black dot moves relative to the red cross. Is the error the same everywhere on the map?
- **D. Design and optimisation challenge.** Design: you can place a third station at (50, 0), (100, 90) or (40, 90) next to A (0, 0) and B (100, 0). Which choices let you find an epicentre at (60, 35) km, and which fail? What is the minimum number of stations for an epicentre from S-P times, and why?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("vp/vs = sqrt(3) = 1.732 for Poisson's ratio 0.25", abs(L.vp_vs_ratio(0.25) - np.sqrt(3.0)) < 1e-12))
    checks.append(("typical speeds 6.0 / 3.5 = 1.714 are within 2 percent of sqrt(3)", abs(L.VP / L.VS - np.sqrt(3.0)) / np.sqrt(3.0) < 0.02))
    try:
        L.vp_vs_ratio(0.5)
        bad = False
    except ValueError:
        bad = True
    checks.append(("Poisson's ratio 0.5 (fluid limit) is rejected", bad))
    d = L.station_distances(L.STATIONS, L.QUAKE)
    checks.append(("worked example: distances are 69.46, 53.15, 58.52 km",
                   np.allclose(d, [69.462, 53.151, 58.523], atol=1e-3)))
    tp, ts = L.travel_time_curves(d)
    checks.append(("worked example: S-P gap at station A is 8.27 s and gives 69.5 km",
                   abs((ts - tp)[0] - 8.269) < 1e-3 and abs(float(L.distance_sp(ts[0], tp[0])) - d[0]) < 1e-9
                   and abs(8.27 * 8.4 - 69.5) < 0.05))
    est = L.trilaterate(L.STATIONS, L.distance_sp(ts, tp))
    checks.append(("worked example: three circles recover the epicentre (60, 35) km", np.allclose(est, [60.0, 35.0], atol=1e-9)))
    # hand calculation of the first row: 200 x = 12000 -> x = 60
    r2 = d ** 2
    checks.append(("hand calculation: first line is 200 x = 12000, so x = 60",
                   abs((r2[0] - r2[1] + 100.0 ** 2) - 12000.0) < 1e-6))
    pts = L.circle_intersections(L.STATIONS[0], d[0], L.STATIONS[1], d[1])
    checks.append(("two stations give two points: (60, 35) and its mirror (60, -35)",
                   len(pts) == 2 and any(np.allclose(p, [60, 35], atol=1e-6) for p in pts)
                   and any(np.allclose(p, [60, -35], atol=1e-6) for p in pts)))
    mirror = np.array([60.0, -35.0])
    checks.append(("the third station rejects the mirror point (126.6 km away, not 58.5)",
                   abs(np.linalg.norm(L.STATIONS[2] - mirror) - 126.6) < 0.1))
    checks.append(("circles that do not touch give no intersection; tangent circles give one",
                   L.circle_intersections((0, 0), 10, (100, 0), 10) == [] and len(L.circle_intersections((0, 0), 10, (20, 0), 10)) == 1))
    try:
        L.trilaterate([[0, 0], [50, 0], [100, 0]], [10.0, 40.0, 90.0])
        bad = False
    except ValueError:
        bad = True
    checks.append(("three stations on one line cannot fix the epicentre (rejected)", bad))
    g = ts - tp
    g[0] += 0.5
    off = L.trilaterate(L.STATIONS, L.distance_sp(g, 0.0))
    checks.append(("a 0.5 s error in one S-P gap moves the epicentre by about 3.6 km",
                   abs(np.linalg.norm(off - L.QUAKE) - 3.61) < 0.05))
    checks.append(("S-P gaps at the three stations are 8.27, 6.33 and 6.97 s", np.allclose(ts - tp, [8.269, 6.327, 6.967], atol=6e-3)))
    checks.append(("hand calculation: second line is 80 x + 180 y = 11100, solved by (60, 35)",
                   abs((r2[0] - r2[2] + 40.0 ** 2 + 90.0 ** 2) - 11100.0) < 1e-6 and abs(80 * 60 + 180 * 35 - 11100) < 1e-9))
    near = np.array([10.0, 10.0])
    g2 = L.station_distances(L.STATIONS, near) * (1 / L.VS - 1 / L.VP)
    g2[0] += 1.0
    est2 = L.trilaterate(L.STATIONS, L.distance_sp(g2, 0.0))
    checks.append(("a 1.0 s error at station A: quake at (10, 10) is off by 1.9 km, quake at (60, 35) by 7.4 km",
                   abs(np.linalg.norm(est2 - near) - 1.9) < 0.1))
    g3 = ts - tp
    g3[0] += 1.0
    e3 = L.trilaterate(L.STATIONS, L.distance_sp(g3, 0.0))
    checks.append(("(60, 35) with a 1.0 s error is off by 7.4 km", abs(np.linalg.norm(e3 - L.QUAKE) - 7.44) < 0.05))
    checks.append(("Poisson's ratio 0.242 reproduces 6.0 / 3.5", abs(L.vp_vs_ratio(0.242) - 6.0 / 3.5) < 1e-3))
    on_line = np.array([[0.0, 0.0], [100.0, 0.0], [50.0, 0.0]])
    try:
        L.trilaterate(on_line, [69.0, 53.0, 40.0])
        bad = False
    except ValueError:
        bad = True
    alt = np.array([[0.0, 0.0], [100.0, 0.0], [100.0, 90.0]])
    est_alt = L.trilaterate(alt, L.station_distances(alt, L.QUAKE))
    checks.append(("design: station (50, 0) is collinear and refused; (100, 90) works and finds (60, 35)",
                   bad and np.allclose(est_alt, [60.0, 35.0], atol=1e-9)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
